# 05 - Feature Engineering

## Electricity Consumption Analysis and Short-Term Demand Forecasting

### Objective

Prepare the hourly electricity dataset for machine learning. The target is **next-hour electricity consumption**. All lag and rolling features use only past information to avoid data leakage.


## 1. Import Libraries


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)


## 2. Load the Hourly Dataset


In [2]:
file_path = "../data/processed/hourly_electricity_consumption.csv"

df = pd.read_csv(file_path, parse_dates=["DateTime"])
df = df.set_index("DateTime").sort_index()

print("Original shape:", df.shape)
df.head()

Original shape: (24584, 15)


,Electricity_Consumption,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Year,Month,Day,Hour,DayOfWeek,IsWeekend,Quarter,DayName
DateTime,,,,,,,,,,,,,,,
2006-12-16 17:00:00,4.222889,0.229000,234.643889,18.100000,0.0,0.527778,16.861111,2006,12,16,17,5,1,4,Saturday
2006-12-16 18:00:00,3.632200,0.080033,234.580167,15.600000,0.0,6.716667,16.866667,2006,12,16,18,5,1,4,Saturday
2006-12-16 19:00:00,3.400233,0.085233,233.232500,14.503333,0.0,1.433333,16.683333,2006,12,16,19,5,1,4,Saturday
2006-12-16 20:00:00,3.268567,0.075100,234.071500,13.916667,0.0,0.000000,16.783333,2006,12,16,20,5,1,4,Saturday
2006-12-16 21:00:00,3.056467,0.076667,237.158667,13.046667,0.0,0.416667,17.216667,2006,12,16,21,5,1,4,Saturday


## 3. Check the Time Series


In [3]:
print("Start:", df.index.min())
print("End:", df.index.max())
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())


Start: 2006-12-16 17:00:00
End: 2009-10-06 00:00:00
Missing values: 0
Duplicate rows: 0


## 4. Keep the Main Target Variable

We use `Electricity_Consumption` as the forecasting target. We do not automatically use future electrical measurements such as future voltage or future global intensity because those values would generally not be known when making a real forecast.


In [4]:
target_column = "Electricity_Consumption"
model_df = df[[target_column]].copy()
print("Model preparation shape:", model_df.shape)


Model preparation shape: (24584, 1)


## 5. Add Calendar Features


In [5]:
model_df["Hour"] = model_df.index.hour
model_df["DayOfWeek"] = model_df.index.dayofweek
model_df["Month"] = model_df.index.month
model_df["Quarter"] = model_df.index.quarter
model_df["IsWeekend"] = (model_df["DayOfWeek"] >= 5).astype(int)

model_df.head()


,Electricity_Consumption,Hour,DayOfWeek,Month,Quarter,IsWeekend
DateTime,,,,,,
2006-12-16 17:00:00,4.222889,17,5,12,4,1
2006-12-16 18:00:00,3.632200,18,5,12,4,1
2006-12-16 19:00:00,3.400233,19,5,12,4,1
2006-12-16 20:00:00,3.268567,20,5,12,4,1
2006-12-16 21:00:00,3.056467,21,5,12,4,1


## 6. Create Lag Features

- `Lag_1h`: consumption one hour ago
- `Lag_24h`: consumption 24 hours ago
- `Lag_168h`: consumption approximately one week ago


In [6]:
model_df["Lag_1h"] = model_df[target_column].shift(1)
model_df["Lag_24h"] = model_df[target_column].shift(24)
model_df["Lag_168h"] = model_df[target_column].shift(168)

model_df.head(175)


,Electricity_Consumption,Hour,DayOfWeek,Month,Quarter,IsWeekend,Lag_1h,Lag_24h,Lag_168h
DateTime,,,,,,,,,
2006-12-16 17:00:00,4.222889,17,5,12,4,1,NaN,NaN,NaN
2006-12-16 18:00:00,3.632200,18,5,12,4,1,4.222889,NaN,NaN
2006-12-16 19:00:00,3.400233,19,5,12,4,1,3.632200,NaN,NaN
2006-12-16 20:00:00,3.268567,20,5,12,4,1,3.400233,NaN,NaN
2006-12-16 21:00:00,3.056467,21,5,12,4,1,3.268567,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2006-12-23 19:00:00,4.117833,19,5,12,4,1,3.879400,3.938167,3.400233
2006-12-23 20:00:00,4.181400,20,5,12,4,1,4.117833,3.536067,3.268567
2006-12-23 21:00:00,3.288433,21,5,12,4,1,4.181400,4.548667,3.056467


## 7. Create Rolling Historical Features

The `.shift(1)` is intentional. It makes sure the current target value is not included in its own rolling feature.


In [7]:
model_df["Rolling_24h"] = (
    model_df[target_column].shift(1).rolling(window=24).mean()
)

model_df["Rolling_168h"] = (
    model_df[target_column].shift(1).rolling(window=168).mean()
)

model_df[[
    target_column, "Lag_1h", "Lag_24h", "Lag_168h",
    "Rolling_24h", "Rolling_168h"
]].head(175)


,Electricity_Consumption,Lag_1h,Lag_24h,Lag_168h,Rolling_24h,Rolling_168h
DateTime,,,,,,
2006-12-16 17:00:00,4.222889,NaN,NaN,NaN,NaN,NaN
2006-12-16 18:00:00,3.632200,4.222889,NaN,NaN,NaN,NaN
2006-12-16 19:00:00,3.400233,3.632200,NaN,NaN,NaN,NaN
2006-12-16 20:00:00,3.268567,3.400233,NaN,NaN,NaN,NaN
2006-12-16 21:00:00,3.056467,3.268567,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
2006-12-23 19:00:00,4.117833,3.879400,3.938167,3.400233,3.149397,1.772679
2006-12-23 20:00:00,4.181400,4.117833,3.536067,3.268567,3.156883,1.776950
2006-12-23 21:00:00,3.288433,4.181400,4.548667,3.056467,3.183772,1.782384


## 8. Create the Next-Hour Target

For time `t`, the target is electricity consumption at `t + 1 hour`. This creates a one-step-ahead forecasting problem.


In [8]:
model_df["Target_Next_Hour"] = model_df[target_column].shift(-1)

model_df[[
    target_column, "Lag_1h", "Lag_24h", "Lag_168h",
    "Rolling_24h", "Rolling_168h", "Target_Next_Hour"
]].head(10)


,Electricity_Consumption,Lag_1h,Lag_24h,Lag_168h,Rolling_24h,Rolling_168h,Target_Next_Hour
DateTime,,,,,,,
2006-12-16 17:00:00,4.222889,NaN,NaN,NaN,NaN,NaN,3.632200
2006-12-16 18:00:00,3.632200,4.222889,NaN,NaN,NaN,NaN,3.400233
2006-12-16 19:00:00,3.400233,3.632200,NaN,NaN,NaN,NaN,3.268567
2006-12-16 20:00:00,3.268567,3.400233,NaN,NaN,NaN,NaN,3.056467
2006-12-16 21:00:00,3.056467,3.268567,NaN,NaN,NaN,NaN,2.200133
2006-12-16 22:00:00,2.200133,3.056467,NaN,NaN,NaN,NaN,2.061600
2006-12-16 23:00:00,2.061600,2.200133,NaN,NaN,NaN,NaN,1.882467
2006-12-17 00:00:00,1.882467,2.061600,NaN,NaN,NaN,NaN,3.349400
2006-12-17 01:00:00,3.349400,1.882467,NaN,NaN,NaN,NaN,1.587267


## 9. Verify the Forecasting Setup


In [9]:
check_columns = [
    target_column, "Lag_1h", "Lag_24h", "Lag_168h",
    "Rolling_24h", "Rolling_168h", "Target_Next_Hour"
]

model_df[check_columns].iloc[170:175]


,Electricity_Consumption,Lag_1h,Lag_24h,Lag_168h,Rolling_24h,Rolling_168h,Target_Next_Hour
DateTime,,,,,,,
2006-12-23 19:00:00,4.117833,3.879400,3.938167,3.400233,3.149397,1.772679,4.181400
2006-12-23 20:00:00,4.181400,4.117833,3.536067,3.268567,3.156883,1.776950,3.288433
2006-12-23 21:00:00,3.288433,4.181400,4.548667,3.056467,3.183772,1.782384,4.327933
2006-12-23 22:00:00,4.327933,3.288433,3.065067,2.200133,3.131263,1.783764,5.562467
2006-12-23 23:00:00,5.562467,4.327933,2.419200,2.061600,3.183882,1.796430,4.028000


## 10. Check Missing Values Created by Feature Engineering


In [10]:
model_df.isnull().sum()


Electricity_Consumption      0
Hour                         0
DayOfWeek                    0
Month                        0
Quarter                      0
IsWeekend                    0
Lag_1h                       1
Lag_24h                     24
Lag_168h                   168
Rolling_24h                 24
Rolling_168h               168
Target_Next_Hour             1
dtype: int64

## 11. Remove Rows That Cannot Be Used for Modeling

These missing values are created by lag/rolling boundaries and the final next-hour target. They are not new missing measurements in the original cleaned data.


In [11]:
model_df = model_df.dropna().copy()

print("Shape after feature engineering:", model_df.shape)
print("Remaining missing values:", model_df.isnull().sum().sum())


Shape after feature engineering: (24415, 12)
Remaining missing values: 0


## 12. Define Feature and Target Columns


In [12]:
feature_columns = [
    "Hour", "DayOfWeek", "Month", "Quarter", "IsWeekend",
    "Lag_1h", "Lag_24h", "Lag_168h",
    "Rolling_24h", "Rolling_168h"
]

target = "Target_Next_Hour"

X = model_df[feature_columns].copy()
y = model_df[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (24415, 10)
y shape: (24415,)


## 13. Inspect the Modeling Dataset


In [13]:
X.head()


,Hour,DayOfWeek,Month,Quarter,IsWeekend,Lag_1h,Lag_24h,Lag_168h,Rolling_24h,Rolling_168h
DateTime,,,,,,,,,,
2006-12-23 17:00:00,17,5,12,4,1,4.349100,1.496800,4.222889,2.934890,1.763888
2006-12-23 18:00:00,18,5,12,4,1,5.452533,2.686967,3.632200,3.099713,1.771207
2006-12-23 19:00:00,19,5,12,4,1,3.879400,3.938167,3.400233,3.149397,1.772679
2006-12-23 20:00:00,20,5,12,4,1,4.117833,3.536067,3.268567,3.156883,1.776950
2006-12-23 21:00:00,21,5,12,4,1,4.181400,4.548667,3.056467,3.183772,1.782384


In [14]:
y.head()


DateTime
2006-12-23 17:00:00    3.879400
2006-12-23 18:00:00    4.117833
2006-12-23 19:00:00    4.181400
2006-12-23 20:00:00    3.288433
2006-12-23 21:00:00    4.327933
Name: Target_Next_Hour, dtype: float64

## 14. Feature Statistics


In [15]:
X.describe().T


,count,mean,std,min,25%,50%,75%,max
Hour,24415.0,11.502437,6.922915,0.000000,6.000000,12.000000,18.000000,23.000000
DayOfWeek,24415.0,3.000573,2.002497,0.000000,1.000000,3.000000,5.000000,6.000000
Month,24415.0,6.178661,3.335876,1.000000,3.000000,6.000000,9.000000,12.000000
Quarter,24415.0,2.392791,1.076670,1.000000,1.000000,2.000000,3.000000,4.000000
IsWeekend,24415.0,0.286340,0.452060,0.000000,0.000000,0.000000,1.000000,1.000000
Lag_1h,24415.0,1.078942,0.922303,0.124000,0.322733,0.757133,1.568883,6.560533
Lag_24h,24415.0,1.080816,0.924162,0.124000,0.322817,0.758667,1.571117,6.560533
Lag_168h,24415.0,1.084129,0.926206,0.124000,0.323250,0.762767,1.575250,6.560533
Rolling_24h,24415.0,1.079879,0.448590,0.172947,0.788334,1.057519,1.319803,3.501724
Rolling_168h,24415.0,1.081361,0.344643,0.178885,0.840749,1.070406,1.339798,2.100290


## 15. Target Statistics


In [16]:
y.describe()


count    24415.000000
mean         1.078596
std          0.921650
min          0.124000
25%          0.322733
50%          0.756933
75%          1.568550
max          6.560533
Name: Target_Next_Hour, dtype: float64

## 16. Verify Chronological Order


In [17]:
print("Chronological order:", model_df.index.is_monotonic_increasing)
print("Start:", model_df.index.min())
print("End:", model_df.index.max())


Chronological order: True
Start: 2006-12-23 17:00:00
End: 2009-10-05 23:00:00


## 17. Save the Feature-Engineered Dataset


In [18]:
output_path = "../data/processed/modeling_dataset.csv"

model_df.to_csv(output_path, index_label="DateTime")

print("Modeling dataset saved to:")
print(output_path)


Modeling dataset saved to:
../data/processed/modeling_dataset.csv


## 18. Final Validation


In [19]:
print("Final modeling dataset shape:", model_df.shape)
print("Start:", model_df.index.min())
print("End:", model_df.index.max())
print("Remaining missing values:", model_df.isnull().sum().sum())
print("Feature count:", len(feature_columns))
print("Target:", target)

print("\nFeatures:")
for feature in feature_columns:
    print("-", feature)

Final modeling dataset shape: (24415, 12)
Start: 2006-12-23 17:00:00
End: 2009-10-05 23:00:00
Remaining missing values: 0
Feature count: 10
Target: Target_Next_Hour

Features:
- Hour
- DayOfWeek
- Month
- Quarter
- IsWeekend
- Lag_1h
- Lag_24h
- Lag_168h
- Rolling_24h
- Rolling_168h


## Conclusion

The hourly data is now ready for forecasting. We created calendar features, historical lags, leakage-safe rolling averages, and a next-hour target.

### Next notebook

**06 - Time-Based Train/Test Split**

We will split the data chronologically and establish a baseline before training machine-learning models.
